# 01 - Build Unified Dataset

## Objetivo

Validar os dados brutos do Instacart e construir um dataset unificado em nível `pedido-produto` para a etapa de EDA.

Este notebook cobre:

- Leitura dos arquivos brutos em `data/raw`.
- Validações de schema, cardinalidade e integridade das chaves.
- União dos dados `prior` e `train`.
- Enriquecimento com informações de pedidos, produtos, aisles e departments.
- Salvamento do dataset processado em `data/processed`.

Este notebook não constrói o dataset de modelagem em nível `user_id-product_id`.

## Inputs

- `data/raw/orders.csv`
- `data/raw/order_products__prior.csv`
- `data/raw/order_products__train.csv`
- `data/raw/products.csv`
- `data/raw/aisles.csv`
- `data/raw/departments.csv`

## Outputs

- `data/processed/orders_product_unified.parquet`
- `data/test_sample/orders_product_sample.parquet`

## Granularidade

Cada linha representa um produto dentro de um pedido.

Portanto, `order_id` pode se repetir e `eval_set` será mantido para diferenciar `prior` e `train`.

---

## 📋 1. Visão Geral da Base

In [1]:
from pathlib import Path

import pandas as pd

# Configuracoes de visualizacao
pd.set_option("display.max_columns", None)

In [2]:
PROJECT_ROOT = Path("..").resolve()

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

ORDERS_PATH = RAW_DATA_DIR / "orders.csv"
ORDER_PRIOR_PATH = RAW_DATA_DIR / "order_products__prior.csv"
ORDER_TRAIN_PATH = RAW_DATA_DIR / "order_products__train.csv"
PRODUCTS_PATH = RAW_DATA_DIR / "products.csv"
AISLES_PATH = RAW_DATA_DIR / "aisles.csv"
DEPARTMENTS_PATH = RAW_DATA_DIR / "departments.csv"

UNIFIED_DATASET_PATH = PROCESSED_DATA_DIR / "orders_product_unified.parquet"

TEST_SAMPLE_DATA_DIR = PROJECT_ROOT / "data" / "test_sample"
UNIFIED_SAMPLE_PATH = TEST_SAMPLE_DATA_DIR / "orders_product_sample.parquet"

In [3]:
input_paths = {
    "orders": ORDERS_PATH,
    "order_prior": ORDER_PRIOR_PATH,
    "order_train": ORDER_TRAIN_PATH,
    "Products": PRODUCTS_PATH,
    "aisles": AISLES_PATH,
    "departments": DEPARTMENTS_PATH,
}

for dataset_name, path in input_paths.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Arquivo {dataset_name} não encontrado no caminho {path}"
        )

print("Todos os arquivos de entrada foram encontrados.")

Todos os arquivos de entrada foram encontrados.


### 1.1 Carregamento de arquivos

In [4]:
df_orders = pd.read_csv(ORDERS_PATH)

df_order_prior = pd.read_csv(ORDER_PRIOR_PATH)
df_order_train = pd.read_csv(ORDER_TRAIN_PATH)

df_products = pd.read_csv(PRODUCTS_PATH)
df_aisles = pd.read_csv(AISLES_PATH)
df_departments = pd.read_csv(DEPARTMENTS_PATH)

print("Datasets carregados com sucesso.")

Datasets carregados com sucesso.


In [5]:
datasets = {
    "orders": df_orders,
    "order_prior": df_order_prior,
    "order_train": df_order_train,
    "products": df_products,
    "aisles": df_aisles,
    "departments": df_departments,
}

for dataset_name, df in datasets.items():
    print(f"{dataset_name}: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")

orders: 3,421,083 linhas x 7 colunas
order_prior: 32,434,489 linhas x 4 colunas
order_train: 1,384,617 linhas x 4 colunas
products: 49,688 linhas x 4 colunas
aisles: 134 linhas x 2 colunas
departments: 21 linhas x 2 colunas


In [6]:
df_orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [7]:
df_order_prior.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [8]:
df_order_train.head()

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [9]:
df_products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [10]:
df_aisles.head()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [11]:
df_departments.head()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


## Entendimento dos datasets

Os dados brutos do Instacart estão distribuídos em tabelas relacionais.

A tabela `orders` contém o contexto dos pedidos, como usuário, ordem cronológica do pedido, dia da semana, horário e o tipo de conjunto (`eval_set`).

No Instacart, a coluna `eval_set` indica o papel de cada pedido:

- `prior`: histórico de pedidos anteriores dos usuários. Será usado para entender comportamento passado e, futuramente, construir features.
- `train`: próximo pedido conhecido de parte dos usuários. Possui os produtos comprados e poderá ser usado futuramente para construir o target supervisionado.
- `test`: próximo pedido desconhecido de parte dos usuários. Não possui produtos associados nas tabelas `order_products`, portanto será usado apenas em uma etapa futura de inferência/submissão.

As tabelas `order_products__prior` e `order_products__train` contêm os produtos presentes em cada pedido. Cada linha representa um produto dentro de um pedido, com informações como posição no carrinho e indicador de recompra.

A tabela `products` contém o cadastro dos produtos e relaciona cada produto a um `aisle` e a um `department`.

As tabelas `aisles` e `departments` são dimensões auxiliares que descrevem as categorias dos produtos.

## Estratégia de unificação

Nesta etapa, o objetivo é construir um dataset unificado em nível `pedido-produto`.

Para isso, os itens dos pedidos de `prior` e `train` serão combinados e enriquecidos com:

- Informações do pedido a partir de `orders`.
- Informações do produto a partir de `products`.
- Informações de aisle a partir de `aisles`.
- Informações de department a partir de `departments`.

O conjunto `test` não será incluído porque ele possui pedidos sem os produtos comprados, sendo útil apenas para inferência final.

## Finalidade do dataset unificado

O dataset unificado será usado na próxima etapa para EDA transacional.

Com ele, será possível analisar padrões como:

- Volume de pedidos, usuários e produtos.
- Distribuição de itens por pedido.
- Produtos mais comprados.
- Taxa de recompra.
- Comportamento por dia da semana e hora do dia.
- Diferenças entre os conjuntos `prior` e `train`.

**IMPORTANTE**: Este notebook ainda não constrói o dataset de modelagem e não define a estratégia de treinamento.

A construção do dataset em nível `user_id-product_id`, a criação de features, a definição do target e a estratégia de treino serão realizadas após o EDA.

---

## ☑️ 2. Validações

### 2.1 Validação de schema

**Objetivo**: Validar se todos os datasets carregados possuem as colunas esperadas e ajuda a identificar alterações inesperadas nos arquivos brutos.

In [12]:
expected_schemas = {
    "orders": [
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
    ],
    "order_prior": [
        "order_id",
        "product_id",
        "add_to_cart_order",
        "reordered",
    ],
    "order_train": [
        "order_id",
        "product_id",
        "add_to_cart_order",
        "reordered",
    ],
    "products": [
        "product_id",
        "product_name",
        "aisle_id",
        "department_id",
    ],
    "aisles": [
        "aisle_id",
        "aisle",
    ],
    "departments": [
        "department_id",
        "department",
    ],
}

for dataset_name, expected_columns in expected_schemas.items():
    actual_columns = datasets[dataset_name].columns.to_list()

    missing_columns = set(expected_columns) - set(actual_columns)
    unexpected_columns = set(actual_columns) - set(expected_columns)

    assert not missing_columns, (
        f"{dataset_name} possui colunas ausentes: {sorted(missing_columns)}"
    )

    assert not unexpected_columns, (
        f"{dataset_name} possui colunas inesperadas: {sorted(unexpected_columns)}"
    )

print("validação de schema concluída")

validação de schema concluída


### 2.2 Validação de cardinalidade

**Objetivo**: Garantir a unicidade das chaves primárias das tabelas de contexto e dimensão.

In [13]:
cardinality_validation = pd.DataFrame(
    [
        {
            "dataset": "orders",
            "Key": "order_id",
            "rows": len(df_orders),
            "unique_keys": df_orders["order_id"].nunique(),
            "is_unique": df_orders["order_id"].is_unique,
        },
        {
            "dataset": "products",
            "Key": "product_id",
            "rows": len(df_products),
            "unique_keys": df_products["product_id"].nunique(),
            "is_unique": df_products["product_id"].is_unique,
        },
        {
            "dataset": "aisles",
            "Key": "aisle_id",
            "rows": len(df_aisles),
            "unique_keys": df_aisles["aisle_id"].nunique(),
            "is_unique": df_aisles["aisle_id"].is_unique,
        },
        {
            "dataset": "departments",
            "Key": "department_id",
            "rows": len(df_departments),
            "unique_keys": df_departments["department_id"].nunique(),
            "is_unique": df_departments["department_id"].is_unique,
        },
    ]
)

cardinality_validation

,dataset,Key,rows,unique_keys,is_unique
0,orders,order_id,3421083,3421083,True
1,products,product_id,49688,49688,True
2,aisles,aisle_id,134,134,True
3,departments,department_id,21,21,True


In [14]:
assert cardinality_validation["is_unique"].all(), (
    "Forma encontradas chaves duplicadas nas tabelas validadas. Para mais informação checar tabela acima"
)

print("Validação de cardinalidade concluída com sucesso.")

Validação de cardinalidade concluída com sucesso.


### 2.3 Validação de integridade referencial

**Objetivo**: Validar se as chaves estrangeiras nas tabelas transacionais possuem correspondência nas tabelas de contexto e dimensão.

In [15]:
referential_integrity_validation = pd.DataFrame(
    [
        {
            "Key": "order_id",
            "tables": "order_product_prior -> orders",
            "missing_keys": len(
                set(df_order_prior["order_id"]) - set(df_orders["order_id"])
            ),
        },
        {
            "Key": "order_id",
            "tables": "order_product_train -> orders",
            "missing_keys": len(
                set(df_order_train["order_id"]) - set(df_orders["order_id"])
            ),
        },
        {
            "Key": "product_id",
            "tables": "order_product_prior -> products",
            "missing_keys": len(
                set(df_order_prior["product_id"]) - set(df_products["product_id"])
            ),
        },
        {
            "Key": "product_id",
            "tables": "order_product_train -> products",
            "missing_keys": len(
                set(df_order_train["product_id"]) - set(df_products["product_id"])
            ),
        },
        {
            "Key": "aisle_id",
            "tables": "products -> aisles",
            "missing_keys": len(
                set(df_products["aisle_id"]) - set(df_aisles["aisle_id"])
            ),
        },
        {
            "Key": "department_id",
            "tables": "products -> department",
            "missing_keys": len(
                set(df_products["department_id"]) - set(df_departments["department_id"])
            ),
        },
    ]
)

referential_integrity_validation

,Key,tables,missing_keys
0,order_id,order_product_prior -> orders,0
1,order_id,order_product_train -> orders,0
2,product_id,order_product_prior -> products,0
3,product_id,order_product_train -> products,0
4,aisle_id,products -> aisles,0
5,department_id,products -> department,0


In [16]:
assert (referential_integrity_validation["missing_keys"] == 0).all(), (
    "Foram encontradas chaves sem correspondência entre as tabelas."
)

print("Validação de integridade referencial concluída com sucesso.")

Validação de integridade referencial concluída com sucesso.


### 2.4 Validação do `eval_set`

**Objetivo**: 
- Validar se o pedidos presentes nas tabelas `order_products_prior` e `order_products_train` estão corretamentes identificados na tabela `orders`.
- Validar se cada usuário possui no máximo um pedido nos conjuntos `train` e `test`.

In [17]:
prior_order_eval_sets = (
    df_order_prior[["order_id"]]
    .drop_duplicates()
    .merge(
        df_orders[["order_id", "eval_set"]],
        on="order_id",
        how="left",
    )
)

train_order_eval_sets = (
    df_order_train[["order_id"]]
    .drop_duplicates()
    .merge(
        df_orders[["order_id", "eval_set"]],
        on="order_id",
        how="left",
    )
)

invalid_prior_orders = prior_order_eval_sets[
    prior_order_eval_sets["eval_set"] != "prior"
]
invalid_train_orders = train_order_eval_sets[
    train_order_eval_sets["eval_set"] != "train"
]

train_orders_per_user = (
    df_orders[df_orders["eval_set"] == "train"].groupby("user_id")["order_id"].nunique()
)

test_orders_per_user = (
    df_orders[df_orders["eval_set"] == "test"].groupby("user_id")["order_id"].nunique()
)

In [18]:
eval_set_validation = pd.DataFrame(
    [
        {
            "validation": "prior orders marked as prior",
            "invalid_records": len(invalid_prior_orders),
            "is_valid": invalid_prior_orders.empty,
        },
        {
            "validation": "train orders marked as train",
            "invalid_records": len(invalid_train_orders),
            "is_valid": invalid_train_orders.empty,
        },
        {
            "validation": "max one train order per user",
            "invalid_records": int((train_orders_per_user > 1).sum()),
            "is_valid": (train_orders_per_user <= 1).all(),
        },
        {
            "validation": "max one test order per user",
            "invalid_records": int((test_orders_per_user > 1).sum()),
            "is_valid": (test_orders_per_user <= 1).all(),
        },
    ]
)

eval_set_validation

,validation,invalid_records,is_valid
0,prior orders marked as prior,0,True
1,train orders marked as train,0,True
2,max one train order per user,0,True
3,max one test order per user,0,True


In [19]:
assert eval_set_validation["is_valid"].all(), (
    "Foram encontrados problemas na validação do eval_set."
)

print("Validação do eval_set concluída com sucesso.")

Validação do eval_set concluída com sucesso.


---

##  📁​ 3. Construção do dataset unificado

**Objetivo**: Criar a tabela transacional unificada a nível de `pedido-produto` dos pedidos dos conjuntos `prior` e `train`

In [20]:
df_order_products = pd.concat(
    [df_order_prior, df_order_train],
    ignore_index=True,
)

expected_order_products_rows = len(df_order_prior) + len(df_order_train)

assert len(df_order_products) == expected_order_products_rows, (
    "A concatenação de prior e train gerou uma quantidade inesperada de linhas."
)

df_order_products.shape

(33819106, 4)

In [21]:
df_unified = df_order_products.merge(
    df_orders,
    on="order_id",
    how="left",
    validate="many_to_one",
)

assert len(df_unified) == len(df_order_products), (
    "O merge com orders alterou a quantidade de linhas"
)

df_unified["eval_set"].value_counts()

eval_set
prior    32434489
train     1384617
Name: count, dtype: int64

In [22]:
df_unified = df_unified.merge(
    df_products,
    on="product_id",
    how="left",
    validate="many_to_one",
)

assert len(df_unified) == len(df_order_products), (
    "O merge com products alterou a quantidade de linhas"
)

df_unified = df_unified.merge(
    df_aisles,
    on="aisle_id",
    how="left",
    validate="many_to_one",
)

assert len(df_unified) == len(df_order_products), (
    "O merge com orders alterou a quantidade de linhas"
)

df_unified = df_unified.merge(
    df_departments,
    on="department_id",
    how="left",
    validate="many_to_one",
)

assert len(df_unified) == len(df_order_products), (
    "O merge com departments alterou a quantidade de linhas."
)

df_unified.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


---

## ✅ 4. Validação Final

**Objetivo**: Verificar a qualidade final do dataset unificado

In [23]:
key_columns = [
    "order_id",
    "user_id",
    "product_id",
    "aisle_id",
    "department_id",
    "eval_set",
]

missing_keys = df_unified[key_columns].isna().sum()

assert (missing_keys == 0).all(), (
    f"Existem valores nulos em colunas-chave: "
    f"{missing_keys[missing_keys > 0].to_dict()}"
)

duplicated_order_product = df_unified.duplicated(
    subset=["order_id", "product_id"]
).sum()

assert duplicated_order_product == 0, (
    f"Existem duplicidades na granularidade order_id-product_id: "
    f"{duplicated_order_product:,}"
)

print("Validação final estrutural concluída com sucesso.")

Validação final estrutural concluída com sucesso.


---

## 💾 5. Persistindo dataset unificado

**Objetivo**: Salvar o dataset unificado em `data/processed` no formato Parquet. Este arquivo representa o artefato processado canônico que será utilizado na etapa de EDA.

O Parquet foi escolhido por ser um formato colunar, comprimido e mais eficiente para leitura analítica do que CSV, especialmente considerando o volume do dataset `prior`. O formato Parquet foi escolhido por ser colunar, comprimido e mais eficiente para leitura analítica do que CSV, especialmente considerando o volume do dataset `prior`.

Além disso, será salva uma amostra reproduzível em `data/test_sample` apenas para testes rápidos e desenvolvimento exploratório. **Essa amostra não deve ser usada como fonte para conclusões analíticas e não faz parte do pipeline oficial versionado pelo DVC**.

In [24]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
TEST_SAMPLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

df_unified.to_parquet(
    UNIFIED_DATASET_PATH,
    index=False,
)

df_unified_sample = df_unified.sample(
    n=min(100_000, len(df_unified)),
    random_state=42,
)
df_unified_sample.to_parquet(UNIFIED_SAMPLE_PATH, index=False)

print(f"Dataset unificado salvo em: {UNIFIED_DATASET_PATH}")
print(f"Amostra para testes salva em: {UNIFIED_SAMPLE_PATH}")

Dataset unificado salvo em: C:\Users\erick\projetos\mlp-market-recommender-system\data\processed\orders_product_unified.parquet
Amostra para testes salva em: C:\Users\erick\projetos\mlp-market-recommender-system\data\test_sample\orders_product_sample.parquet
